In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text

In [12]:
# -----------------------------------
# 1. DATASET (REALISTIC + CONSTRAINTS)
# -----------------------------------
# color: 0=red, 1=yellow, 2=orange, 3=green

X = [
    # Apple (red, green, yellow only)
    [0, 7, 7], [3, 6, 6], [1, 8, 7], [0, 7.5, 7.2],

    # Banana (any color)
    [1, 18, 4], [3, 20, 5], [0, 17, 4], [1, 16, 4],

    # Orange (ONLY orange color)
    [2, 6, 6], [2, 7, 7], [2, 5.5, 5.5], [2, 6.5, 6.5],

    # Mango (yellow / green only)
    [1, 12, 6], [3, 10, 5], [1, 11, 6], [3, 13, 6]
]

# Labels
# 0=Apple, 1=Banana, 2=Orange, 3=Mango
y = [
    0,0,0,0,
    1,1,1,1,
    2,2,2,2,
    3,3,3,3
]

In [3]:
# -----------------------------------
# 2. FEATURE ENGINEERING (RATIO)
# -----------------------------------
X_feat = []

for c, h, w in X:
    h_scaled = int(h * 10)
    w_scaled = int(w * 10)

    # ratio scaled (avoid float)
    ratio = int((w_scaled * 10) / h_scaled)

    X_feat.append([c, h_scaled, w_scaled, ratio])

X_feat = np.array(X_feat)
y = np.array(y)


In [13]:
# -----------------------------------
# 3. TRAIN MODEL
# -----------------------------------
model = RandomForestClassifier(
    n_estimators=4,
    max_depth=3,
    random_state=42
)

model.fit(X_feat, y)

print("Model trained ✅")
print("Training accuracy:", model.score(X_feat, y))

Model trained ✅
Training accuracy: 0.9375


In [14]:
# -----------------------------------
# 4. PRINT TREES (FOR VERILOG)
# -----------------------------------
feature_names = ["color", "height", "width", "ratio"]

for i, tree in enumerate(model.estimators_):
    print(f"\n===== Tree {i} =====")
    print(export_text(tree, feature_names=feature_names))


===== Tree 0 =====
|--- height <= 72.50
|   |--- class: 2.0
|--- height >  72.50
|   |--- color <= 2.00
|   |   |--- width <= 55.00
|   |   |   |--- class: 1.0
|   |   |--- width >  55.00
|   |   |   |--- class: 0.0
|   |--- color >  2.00
|   |   |--- class: 3.0


===== Tree 1 =====
|--- ratio <= 7.00
|   |--- width <= 55.00
|   |   |--- class: 1.0
|   |--- width >  55.00
|   |   |--- class: 3.0
|--- ratio >  7.00
|   |--- color <= 1.00
|   |   |--- class: 0.0
|   |--- color >  1.00
|   |   |--- class: 2.0


===== Tree 2 =====
|--- width <= 52.50
|   |--- class: 1.0
|--- width >  52.50
|   |--- height <= 95.00
|   |   |--- height <= 65.00
|   |   |   |--- class: 2.0
|   |   |--- height >  65.00
|   |   |   |--- class: 0.0
|   |--- height >  95.00
|   |   |--- class: 3.0


===== Tree 3 =====
|--- width <= 52.50
|   |--- class: 1.0
|--- width >  52.50
|   |--- color <= 1.00
|   |   |--- class: 0.0
|   |--- color >  1.00
|   |   |--- height <= 92.50
|   |   |   |--- class: 2.0
|   |   |-

In [15]:
# -----------------------------------
# 5. TEST FUNCTION
# -----------------------------------
def predict_fruit(color, height, width):
    h_scaled = int(height * 10)
    w_scaled = int(width * 10)

    ratio = int((w_scaled * 10) / h_scaled)

    X_test = np.array([[color, h_scaled, w_scaled, ratio]])

    pred = model.predict(X_test)[0]

    fruit_map = {
        0: "Apple",
        1: "Banana",
        2: "Orange",
        3: "Mango"
    }

    return fruit_map[pred]

In [22]:
# -----------------------------------
# 6. TEST CASES
# -----------------------------------
print("\n--- TESTING ---")

tests = [
    (1, 18, 4),   # banana
    (0, 7, 7),    # apple
    (2, 6, 6),    # orange
    (3, 12, 6),   # mango
    (1, 20, 3),   # strong banana
    (0, 6, 5.5),  # apple-ish
    (0, 10, 7),   # mango-ish apple
    (2, 11, 7),   # mango ish mango
]

for c, h, w in tests:
    print(f"Input: c={c}, h={h}, w={w} → {predict_fruit(c,h,w)}")


--- TESTING ---
Input: c=1, h=18, w=4 → Banana
Input: c=0, h=7, w=7 → Apple
Input: c=2, h=6, w=6 → Orange
Input: c=3, h=12, w=6 → Mango
Input: c=1, h=20, w=3 → Banana
Input: c=0, h=6, w=5.5 → Apple
Input: c=0, h=10, w=7 → Apple
Input: c=2, h=11, w=7 → Mango
